# Pet document categorisation

**What this notebook is for:** every file uploaded to a pet claim, not only patient histories. Three sections:

1. What the file table looks like: counts of content types, folders, upload sources and deleted flags.
2. The 100 files behind the label shortlist: the sample Gemini described, from which `DOCUMENT_TYPES` was derived.
3. Evaluation sample for the sorter: fresh files that Gemini labels and a person labels by hand, then compared.

**Kernel:** use `../vet-record-reader/.venv/bin/python`, the environment of the first notebook; it has
every package needed, so there is no second environment.

**Before the first run:** `gcloud auth application-default login` (signs this machine in to Google Cloud).


In [ ]:
# cell: bq
import pandas as pd
from google.cloud import bigquery

# BigQuery client (your own Google login) and a query helper that dry-runs first
bq_client = bigquery.Client(project='hedvig-coruscant')


def run_query(sql, max_gb=10):
    dry = bq_client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    gb = dry.total_bytes_processed / 1e9
    print(f'Dry run: this query scans {gb:.2f} GB')
    if gb > max_gb:
        raise ValueError(f'{gb:.2f} GB is above max_gb={max_gb}. Approve by raising max_gb.')
    return bq_client.query(sql).result().to_dataframe()

## What the file table looks like

Counts over all pet claim files since 2025 in `dbt_prod.stg_claims_service__claim_file`. Run 19 Nov 2026: 36,280 files on
8,149 claims; content type PDF 18,187 / JPEG 14,010 / JSON 2,302 / PNG 1,574, the rest under 30 each; `category` always
empty; `folder` empty for 94%; `deleted_at` is the only deleted column ever set (1,637 files), so queries filter on it.


In [ ]:
# cell: file-counts
# Counts over all pet claim files since 2025: what values content_type, category, folder, upload_source
# and the three deleted columns take, with files and claims per value
SQL = """
WITH pet_files AS (
  SELECT f.*
  FROM `hedvig-coruscant.dbt_prod.stg_claims_service__claim_file` f
  JOIN (SELECT DISTINCT claim_id FROM `hedvig-coruscant.dbt_prod.claims_reporting`
        WHERE promise_category = 'Pet' AND acquisition_source != 'manypets') cr
    ON cr.claim_id = f.claims_id
  WHERE DATE(f.uploaded_at) > '2025-01-01'
)
SELECT 'content_type' AS what, content_type AS value, COUNT(*) AS files, COUNT(DISTINCT claims_id) AS claims FROM pet_files GROUP BY 2
UNION ALL SELECT 'category', category, COUNT(*), COUNT(DISTINCT claims_id) FROM pet_files GROUP BY 2
UNION ALL SELECT 'folder', folder, COUNT(*), COUNT(DISTINCT claims_id) FROM pet_files GROUP BY 2
UNION ALL SELECT 'upload_source', upload_source, COUNT(*), COUNT(DISTINCT claims_id) FROM pet_files GROUP BY 2
UNION ALL SELECT 'deleted flags',
  CONCAT('deleted_at=', CAST(deleted_at IS NOT NULL AS STRING),
         ' marked=', CAST(marked_as_deleted_at IS NOT NULL AS STRING),
         ' markd=', CAST(markd_as_deleted_at IS NOT NULL AS STRING)),
  COUNT(*), COUNT(DISTINCT claims_id) FROM pet_files GROUP BY 2
ORDER BY what, files DESC
"""
file_counts = run_query(SQL)
pd.set_option("display.max_rows", 200)
file_counts


## The 100 files behind the label shortlist

`shortlist_data_sample.csv` (in this folder, described in `shortlist_data_sample.md`) is the sample of 100 pet-claim
files that Gemini described in its own words on 19 Nov 2026. The `DOCUMENT_TYPES` label list in
`../vet-record-reader/contract_journal_reader.ipynb` was derived from these descriptions. Nothing is re-run here.


In [ ]:
# cell: shortlist-sample
shortlist = pd.read_csv("shortlist_data_sample.csv")
print(f"{len(shortlist)} files; Gemini's own labels (kind):")
print(shortlist["kind"].value_counts().to_string())
pd.set_option("display.max_colwidth", 120)
shortlist[["file_name", "content_type", "upload_source", "what_it_is", "kind", "summary"]].sort_values("kind")


## Evaluation sample for the sorter

Checks stage 2 of `contract_journal_reader.ipynb`: Gemini labels each file with one or more of `DOCUMENT_TYPES`.
Steps: draw `EVAL_SAMPLE_SIZE` pet files that are not in the 100-file shortlist sample; let Gemini label them
(`eval-sort`, about $0.004 per file); a person fills in `hand_primary_type` and `hand_document_types` in
`output/eval_sample_labels.csv` (several labels separated by `;`); `eval-score` compares the two.


In [ ]:
# cell: eval-config
from pathlib import Path

EVAL_SAMPLE_SIZE = 50

# --- Google Cloud / Gemini (same settings as contract_journal_reader.ipynb) ---
GEMINI_PROJECT = "hedvig-coruscant"
GEMINI_LOCATION = "eu"
GEMINI_MODEL = "gemini-3.8-flash"
TEMPERATURE = 0
LABELS = {"project": "vet-record-reader", "team": "maker-days-2026"}   # shown against the cost in Cloud Billing

# Prices for gemini-3.8-flash on the Vertex AI EU endpoint, USD per million tokens. Third-party listings
# (requesty.ai, sim.ai), read 18 Nov 2026; promotional rates until 31 Dec 2026, listed to double from 1 Jan 2027.
PRICE_PER_M_INPUT_USD = 0.83
PRICE_PER_M_OUTPUT_USD = 4.13        # thinking tokens are billed as output

# --- AWS (files are on S3 in the prod account) ---
AWS_REGION = "eu-central-1"
SOURCE_PROFILE = "ops"               # your own user, in ~/.aws
ROLE_ARN = "arn:aws:iam::658549670687:role/CrossAccountReadOnlyRole"
CREDENTIALS_CACHE = Path("/tmp/.vet-record-reader-aws-credentials.json")   # shared with the other notebook
SESSION_HOURS = 1                    # the role allows 1 hour today; 12 once #platform raises its maximum

# --- The document types Gemini may choose from (derived from the 100-file sample, 19 Nov) ---
DOCUMENT_TYPES = {
    "medical_record": "veterinary journal or patient history: what the vet observed, diagnosed and did during a visit",
    "discharge_instructions": "aftercare advice to the owner after a visit: medication schedule, diet, rest, follow-up",
    "invoice": "a bill listing treatments and the amount owed, including a clinic's billing specification or draft invoice",
    "receipt": "proof that payment was made: kvitto, card payment slip, pharmacy receipt",
    "direct_settlement_request": "the clinic's request that Hedvig pay it directly (direktreglering), or a claim form",
    "lab_or_imaging_report": "laboratory results, blood analyser readout, X-ray or other diagnostic image",
    "veterinary_certificate": "health or inspection certificate (besiktningsintyg), consent form, referral",
    "prescription_or_medication": "a prescription slip, or a photo of a medicine package with its label",
    "insurance_policy": "an insurance policy document",
    "pet_passport": "pages from a pet passport",
    "photo_of_animal": "a photograph of the pet itself, showing the injury or symptom",
    "other": "none of the above, for example a clinic logo image",
}


In [ ]:
# cell: eval-helpers
import json
import time
from typing import Literal

import boto3
from google import genai
from google.genai import types
from pydantic import BaseModel


# ------------------------------------------- S3 -------------------------------------------
def assume_read_only_role():
    """Temporary keys as CrossAccountReadOnlyRole, valid SESSION_HOURS. Asks for your MFA code once."""
    ops = boto3.Session(profile_name=SOURCE_PROFILE)
    mfa_serial = ops._session.full_config["profiles"].get("prod", {}).get("mfa_serial")
    kwargs = {"RoleArn": ROLE_ARN, "RoleSessionName": "vet-record-reader", "DurationSeconds": SESSION_HOURS * 3600}
    if mfa_serial:
        kwargs |= {"SerialNumber": mfa_serial, "TokenCode": input("AWS MFA code: ")}
    c = ops.client("sts").assume_role(**kwargs)["Credentials"]
    CREDENTIALS_CACHE.write_text(json.dumps({
        "AccessKeyId": c["AccessKeyId"], "SecretAccessKey": c["SecretAccessKey"],
        "SessionToken": c["SessionToken"], "Expiration": c["Expiration"].timestamp(),
    }))
    print(f"AWS credentials valid until {c['Expiration']:%Y-%m-%d %H:%M %Z}")


def prod_session():
    if not CREDENTIALS_CACHE.exists() or json.loads(CREDENTIALS_CACHE.read_text())["Expiration"] < time.time() + 60:
        assume_read_only_role()
    c = json.loads(CREDENTIALS_CACHE.read_text())
    return boto3.Session(aws_access_key_id=c["AccessKeyId"], aws_secret_access_key=c["SecretAccessKey"],
                         aws_session_token=c["SessionToken"], region_name=AWS_REGION)


def fetch_file(bucket, key):
    """The file's bytes. Read-only: nothing in S3 is written, moved or deleted."""
    return prod_session().client("s3").get_object(Bucket=bucket, Key=key)["Body"].read()


# ----------------------------------------- Gemini -----------------------------------------
gemini = genai.Client(vertexai=True, project=GEMINI_PROJECT, location=GEMINI_LOCATION)
CALLS = []   # one row per Gemini call: stage, label, tokens, cost


def ask_gemini(files, prompt, stage, label, output_format=None, model=GEMINI_MODEL):
    """Send one or more files and a prompt to Gemini in one call. Same function as in contract_journal_reader.ipynb.

    files: list of (file_label, bytes, content_type). Each file is preceded by a text part naming it.
    output_format: a pydantic class the answer must fit, or None for free text.
    Returns (answer, usage). Falls back to JSON-in-the-prompt if Gemini rejects the forced shape.
    """
    parts = []
    for file_label, data, content_type in files:
        parts += [f"[{file_label}]", types.Part.from_bytes(data=data, mime_type=content_type)]
    config = types.GenerateContentConfig(temperature=TEMPERATURE, labels=LABELS)
    forced = output_format is not None
    if forced:
        config.response_mime_type = "application/json"
        config.response_schema = output_format
    try:
        response = gemini.models.generate_content(model=model, contents=parts + [prompt], config=config)
    except Exception as e:
        if not forced or "schema" not in str(e).lower():
            raise
        print(f"Gemini rejected the forced output shape ({str(e)[:120]}...). Asking for JSON in the prompt instead.")
        config.response_schema = None
        schema_text = json.dumps(output_format.model_json_schema())
        response = gemini.models.generate_content(
            model=model, contents=parts + [prompt + "\n\nAnswer with JSON only, matching this JSON schema:\n" + schema_text],
            config=config)
    u = response.usage_metadata
    usage = {"stage": stage, "label": label, "model": model,
             "input_tokens": u.prompt_token_count, "output_tokens": u.candidates_token_count,
             "thinking_tokens": u.thoughts_token_count or 0, "total_tokens": u.total_token_count}
    usage["cost_usd"] = round((usage["input_tokens"] * PRICE_PER_M_INPUT_USD
                               + (usage["output_tokens"] + usage["thinking_tokens"]) * PRICE_PER_M_OUTPUT_USD) / 1e6, 4)
    CALLS.append(usage)
    if forced:
        answer = response.parsed if response.parsed is not None else output_format.model_validate_json(response.text)
    else:
        answer = response.text
    return answer, usage


# ------------------------------- The sorter (stage 2, copied) -------------------------------
DocumentType = Literal[tuple(DOCUMENT_TYPES)]


class FileType(BaseModel):
    document_types: list[DocumentType]   # every kind of content in the file
    primary_type: DocumentType           # the one that best describes the file as a whole
    confidence: Literal["high", "medium", "low"]
    one_line_summary: str
    language: str


SORT_PROMPT = (
    "You are sorting a file uploaded to a pet insurance claim. A file often holds several kinds of content "
    "(for example a clinic's direct settlement bundle has a journal, an invoice draft and discharge instructions). "
    "List every type present in the file, choosing only from these types:\n"
    + "\n".join(f"- {k}: {v}" for k, v in DOCUMENT_TYPES.items())
    + "\n\nThen give the one primary type that best describes the file as a whole, your confidence, a one-line "
    "summary of what the file contains (no owner personal data), and the language of the document. "
    "Use only what is in the file."
)


In [ ]:
# cell: eval-sample
# Same filters as the shortlist sample, minus the 100 files already in it, same fixed order
shortlist_ids = ", ".join(f"'{i}'" for i in pd.read_csv("shortlist_data_sample.csv")["id"])

SQL = f"""
SELECT f.id, f.claims_id, f.file_name, f.content_type, f.upload_source, f.uploaded_at, f.bucket, f.key
FROM hedvig-coruscant.dbt_prod.stg_claims_service__claim_file f
JOIN (
SELECT DISTINCT claim_id
FROM hedvig-coruscant.dbt_prod.claims_reporting
WHERE promise_category = 'Pet'
AND acquisition_source != 'manypets'
) cr ON cr.claim_id = f.claims_id
WHERE DATE(f.uploaded_at) > '2025-01-01'
AND f.deleted_at IS NULL
AND f.content_type IN ('application/pdf', 'image/jpeg', 'image/png')
AND CAST(f.id AS STRING) NOT IN ({shortlist_ids})
ORDER BY FARM_FINGERPRINT(CAST(f.id AS STRING))
LIMIT {EVAL_SAMPLE_SIZE}
"""
eval_files = run_query(SQL)
print(len(eval_files), "files")
eval_files["content_type"].value_counts()


In [ ]:
# cell: eval-sort
rows = []
for _, r in eval_files.iterrows():
    base = r[["id", "claims_id", "file_name", "content_type", "upload_source", "uploaded_at"]].to_dict()
    try:
        data = fetch_file(r["bucket"], r["key"])
        answer, usage = ask_gemini([("file", data, r["content_type"])], SORT_PROMPT,
                                   stage="eval sort", label=r["file_name"], output_format=FileType)
        rows.append({**base, **answer.model_dump(), "document_types": ";".join(answer.document_types),
                     "size_mb": round(len(data) / 1e6, 2), "cost_usd": usage["cost_usd"]})
        print(f"{r['file_name'][:50]:50} | {answer.primary_type:26} | {';'.join(answer.document_types)}")
    except Exception as e:
        rows.append({**base, "primary_type": "ERROR", "document_types": "", "one_line_summary": str(e)[:200]})
        print(f"{r['file_name'][:50]:50} | ERROR: {str(e)[:80]}")

eval_labels = pd.DataFrame(rows)
eval_labels["hand_primary_type"] = ""      # a person fills these in: one label, and labels separated by ;
eval_labels["hand_document_types"] = ""
eval_labels["note"] = ""
Path("output").mkdir(exist_ok=True)
eval_labels.to_csv("output/eval_sample_labels.csv", index=False)
print(f"\n{len(eval_labels)} files, ${eval_labels['cost_usd'].sum():.2f}; saved output/eval_sample_labels.csv")
eval_labels["primary_type"].value_counts()


In [ ]:
# cell: eval-score
# Reads the csv back after a person has filled in hand_primary_type / hand_document_types
scored = pd.read_csv("output/eval_sample_labels.csv").fillna("")
scored = scored[scored["hand_primary_type"] != ""]
print(f"{len(scored)} files labelled by hand")

# (a) primary type: share of files where Gemini's primary_type equals the hand label
agree = (scored["primary_type"] == scored["hand_primary_type"]).mean()
print(f"primary_type agrees with the hand label on {agree:.0%} of {len(scored)} files\n")

# (b) per label over the full label lists: of files Gemini gave the label, share a person also gave it (precision);
#     of files a person gave the label, share Gemini also gave it (recall)
split = lambda s: set(x.strip() for x in s.split(";") if x.strip())
g = scored["document_types"].apply(split)
h = scored["hand_document_types"].apply(split)
per_label = []
for label in DOCUMENT_TYPES:
    gemini_has = g.apply(lambda s: label in s)
    hand_has = h.apply(lambda s: label in s)
    both = (gemini_has & hand_has).sum()
    per_label.append({"label": label, "gemini_gave_it": gemini_has.sum(), "hand_gave_it": hand_has.sum(),
                      "share_of_gemini's_also_by_hand": round(both / gemini_has.sum(), 2) if gemini_has.sum() else None,
                      "share_of_hand's_also_by_gemini": round(both / hand_has.sum(), 2) if hand_has.sum() else None})
display(pd.DataFrame(per_label))

# (c) Gemini primary type (rows) against hand primary type (columns)
pd.crosstab(scored["primary_type"], scored["hand_primary_type"])
